# VN Traffic AI — Final Report Notebook

**Mục đích**: 1 notebook duy nhất chạy toàn bộ bước còn lại — eval, demo, visualize, benchmark — xuất artifacts cho báo cáo cuối kỳ.

**Model chủ đạo**: `weights/v3_vn_best.pt` (mAP50=0.951 trên val VN).

**Pre-requisite**: Đã chạy xong `scripts/data_prep/merge_v3.py` + `scripts/training/v3_vn_only.py`.

**Chạy toàn bộ**: Kernel → Restart & Run All (~20-30 phút tổng).

---

## Mục lục
1. [Setup + imports](#1)
2. [Dataset exploration](#2)
3. [Model inference demo](#3)
4. [Full evaluation 5 models](#4)
5. [Error analysis chi tiết (per-class + per-source + per-size)](#5)
6. [Failure case analysis](#6)
7. [Training curves comparison V1→V2→V3→V3-1280](#7)
8. [Pipeline demo end-to-end](#8)
9. [CV pipeline improvement: bottom-center crossing](#9)
10. [Visualize counting events](#10)
11. [FPS benchmark](#11)
12. [Final summary + artifacts](#12)

## 1. Setup + imports <a id='1'></a>

In [ ]:
import os
from pathlib import Path
import json
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

# Working dir = project root
ROOT = Path("/home/xuand/vn_traffic_ai")
os.chdir(ROOT)
print(f"CWD: {Path.cwd()}")

random.seed(42)
np.random.seed(42)

# Config matplotlib
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})


In [ ]:
# Weights — 6 model để so sánh
WEIGHTS = {
    "yolov8s_coco":  str(ROOT / "yolov8s.pt"),                     # COCO vanilla (80 class)
    "V1_baseline":   str(ROOT / "weights/trained/v1_baseline.pt"), # DETRAC-heavy
    "V2_fix":        str(ROOT / "weights/trained/v2_fix.pt"),      # fine-tune + aug
    "V3_vn":         str(ROOT / "weights/v3_vn_best.pt"),          # VN-only 20.5k imgsz=640
    "V3_vn_1280":    str(ROOT / "weights/trained/v3_vn_1280.pt"),  # VN-only imgsz=1280
    "V4_vn_cctv":    str(ROOT / "weights/trained/v4_vn_cctv.pt"),  # V3 + vn_cctv 49.7k ← MOST DATA
}
# Model nào không tồn tại path → auto skip ở eval

# Dataset config
DATA_YAML = ROOT / "data/data_v3.yaml"
VAL_IMG_DIR = ROOT / "data/merged_v3/images/val"
VAL_LBL_DIR = ROOT / "data/merged_v3/labels/val"
TEST_IMG_DIR = ROOT / "data/merged_v3/images/test"
TEST_LBL_DIR = ROOT / "data/merged_v3/labels/test"

# Test video
TEST_VIDEO = ROOT / "data/test_videos/raw/demo_traffic.mp4"
VIDEO_GT = ROOT / "data/test_videos/ground_truth/demo_traffic.json"

# Class canonical
CLASS_NAMES = ["motorcycle", "car", "bus", "truck"]
CLASS_COLORS = [(255, 50, 50), (50, 150, 255), (50, 255, 50), (255, 180, 50)]  # BGR

# Output dirs
RESULTS_DIR = ROOT / "results/final_report"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "charts").mkdir(exist_ok=True)
(RESULTS_DIR / "videos").mkdir(exist_ok=True)
(RESULTS_DIR / "tables").mkdir(exist_ok=True)

for name, pt in WEIGHTS.items():
    exists = "✓" if Path(pt).exists() else "✗ MISSING"
    print(f"  {name:<16} {pt}  [{exists}]")

## 1b. (Optional) Resume training nếu chưa xong

Ô dưới chạy training V3-640 và V3-1280 nếu best.pt chưa tồn tại. Có `save_period=1` nên sẽ auto RESUME từ epoch cuối nếu có `last.pt`.

Bỏ qua (không run ô này) nếu đã có weights xong.

In [ ]:
import subprocess

# V3 (640) — ~3h nếu FRESH
if not Path(WEIGHTS["V3_vn"]).exists():
    print("V3-640 chưa có, chạy training...")
    subprocess.run(["python", "scripts/training/v3_vn_only.py"], check=True)
    best_src = ROOT / "runs/detect/train_v8s_v3_baseline/weights/best.pt"
    if best_src.exists():
        import shutil; shutil.copy(best_src, WEIGHTS["V3_vn"])
else:
    print(f"V3-640 OK: {WEIGHTS['V3_vn']}")

# V3-1280 — ~5-8h nếu FRESH, ít hơn nếu RESUME
if not Path(WEIGHTS["V3_vn_1280"]).exists():
    print("V3-1280 chưa có, chạy training...")
    subprocess.run(["python", "scripts/week2/07_train_v3_1280.py"], check=True)
    best_src = ROOT / "runs/detect/train_v8s_v3_1280/weights/best.pt"
    if best_src.exists():
        import shutil; shutil.copy(best_src, WEIGHTS["V3_vn_1280"])
else:
    print(f"V3-1280 OK: {WEIGHTS['V3_vn_1280']}")

print("\nAll weights ready, proceeding to evaluation.")

## 2. Dataset exploration <a id='2'></a>

Hiển thị:
- Vài ảnh mẫu val set
- Phân phối class val/test
- So sánh class distribution V1 (cũ) vs V3 (mới)

In [ ]:
# Sample images from val set
import yaml
cfg = yaml.safe_load(open(DATA_YAML))
print(f"Dataset: {cfg['nc']} class — {cfg['names']}")
print(f"Train: {cfg.get('train', '?')}")
print(f"Val:   {cfg.get('val', '?')}")
print(f"Test:  {cfg.get('test', '?')}")

In [ ]:
def draw_bbox(img, lbl_path, class_names, colors):
    h, w = img.shape[:2]
    img = img.copy()
    if not Path(lbl_path).exists():
        return img
    for line in open(lbl_path):
        parts = line.strip().split()
        if not parts: continue
        c = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
        x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), colors[c], 2)
        cv2.putText(img, class_names[c], (x1, max(y1-5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[c], 2)
    return img

# Show 6 random val samples
samples = random.sample(sorted(VAL_IMG_DIR.glob("*.jpg")), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.ravel(), samples):
    img = cv2.imread(str(img_path))
    lbl_path = VAL_LBL_DIR / f"{img_path.stem}.txt"
    img = draw_bbox(img, lbl_path, CLASS_NAMES, CLASS_COLORS)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name[:40], fontsize=8)
    ax.axis('off')
plt.suptitle("Val samples — ground truth bbox", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/val_samples.png", dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Class distribution train / val / test
from collections import Counter

def count_classes(lbl_dir):
    cnt = Counter()
    for lbl in Path(lbl_dir).glob("*.txt"):
        for line in open(lbl):
            parts = line.strip().split()
            if parts:
                cnt[int(parts[0])] += 1
    return cnt

splits = {
    "train": count_classes(ROOT / "data/merged_v3/labels/train"),
    "val":   count_classes(VAL_LBL_DIR),
    "test":  count_classes(TEST_LBL_DIR),
}

df_dist = pd.DataFrame({
    split: [cnt.get(i, 0) for i in range(4)]
    for split, cnt in splits.items()
}, index=CLASS_NAMES)
print("Bbox distribution per split:")
display(df_dist)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (split, cnt) in zip(axes, splits.items()):
    total = sum(cnt.values())
    sizes = [cnt.get(i, 0) for i in range(4)]
    ax.pie(sizes, labels=CLASS_NAMES, autopct='%1.1f%%',
           colors=['#E74C3C', '#3498DB', '#27AE60', '#F39C12'])
    ax.set_title(f'{split}\n{total:,} bbox', fontsize=11)
plt.suptitle("V3 — Class distribution per split (VN-only)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/class_distribution.png", dpi=120, bbox_inches='tight')
plt.show()

## 3. Model inference demo <a id='3'></a>

Load V3 model → predict trên 6 ảnh val random → so sánh với ground truth.

In [ ]:
from ultralytics import YOLO

model_v3 = YOLO(WEIGHTS["V3_vn"])
print(f"V3 loaded: {model_v3.names}")

In [ ]:
# Predict on 6 val samples
samples = random.sample(sorted(VAL_IMG_DIR.glob("*.jpg")), 6)
results = model_v3.predict([str(p) for p in samples], conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path, r in zip(axes.ravel(), samples, results):
    img = cv2.imread(str(img_path))
    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().int().tolist()
        c = int(box.cls[0])
        conf = float(box.conf[0])
        cv2.rectangle(img, (x1, y1), (x2, y2), CLASS_COLORS[c], 2)
        cv2.putText(img, f"{CLASS_NAMES[c]} {conf:.2f}", (x1, max(y1-5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, CLASS_COLORS[c], 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{len(r.boxes)} detections", fontsize=10)
    ax.axis('off')
plt.suptitle("V3 predictions on val samples (conf ≥ 0.25)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/v3_predictions_sample.png", dpi=120, bbox_inches='tight')
plt.show()

## 4. Full evaluation V1/V2/V3 vs COCO vanilla <a id='4'></a>

**Fair comparison** — eval cả 4 model trên **cùng val VN-only** (1,779 ảnh), dùng torchmetrics COCO-style mAP.

COCO vanilla có 80 class → filter về 4 vehicle class rồi remap canonical.

In [ ]:
import torch
from torchmetrics.detection import MeanAveragePrecision

# COCO cls (80 class) → canonical (4 class)
COCO_TO_CANON = {2: 1, 3: 0, 5: 2, 7: 3}  # car→1, motor→0, bus→2, truck→3

def load_gt(lbl_path, img_w, img_h):
    boxes, labels = [], []
    if not Path(lbl_path).exists():
        return torch.zeros((0, 4)), torch.zeros((0,), dtype=torch.long)
    for line in open(lbl_path):
        parts = line.strip().split()
        if not parts: continue
        c = int(parts[0])
        xc, yc, w, h = map(float, parts[1:5])
        x1, y1 = (xc - w/2) * img_w, (yc - h/2) * img_h
        x2, y2 = (xc + w/2) * img_w, (yc + h/2) * img_h
        boxes.append([x1, y1, x2, y2])
        labels.append(c)
    return torch.tensor(boxes), torch.tensor(labels, dtype=torch.long)

def eval_model(name, pt_path, is_coco=False, batch_size=32):
    print(f"→ Eval {name}...")
    model = YOLO(pt_path)
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
    img_paths = sorted(VAL_IMG_DIR.glob("*.jpg"))
    for i in range(0, len(img_paths), batch_size):
        batch = img_paths[i:i+batch_size]
        results = model.predict([str(p) for p in batch], imgsz=640,
                                 conf=0.001, iou=0.6, verbose=False, device=0)
        preds, targets = [], []
        for img_path, r in zip(batch, results):
            h, w = r.orig_shape
            if r.boxes is not None and len(r.boxes) > 0:
                boxes = r.boxes.xyxy.cpu()
                scores = r.boxes.conf.cpu()
                labels = r.boxes.cls.cpu().long()
                if is_coco:
                    mask = torch.tensor([int(c) in COCO_TO_CANON for c in labels])
                    boxes, scores, labels = boxes[mask], scores[mask], labels[mask]
                    labels = torch.tensor([COCO_TO_CANON[int(c)] for c in labels],
                                          dtype=torch.long)
            else:
                boxes = torch.zeros((0, 4)); scores = torch.zeros((0,))
                labels = torch.zeros((0,), dtype=torch.long)
            preds.append({"boxes": boxes, "scores": scores, "labels": labels})
            gt_boxes, gt_labels = load_gt(VAL_LBL_DIR / f"{img_path.stem}.txt", w, h)
            targets.append({"boxes": gt_boxes, "labels": gt_labels})
        metric.update(preds, targets)
    r = metric.compute()
    return {
        "mAP50":    float(r["map_50"]),
        "mAP50-95": float(r["map"]),
        "per_class_mAP": {CLASS_NAMES[i]: float(r["map_per_class"][i])
                          for i in range(4)},
    }

In [ ]:
# Run full eval (chậm nhất ~2 phút total)
eval_results = {}
for name, pt in WEIGHTS.items():
    if not Path(pt).exists():
        print(f"  Skip {name}: not found")
        continue
    eval_results[name] = eval_model(name, pt, is_coco=(name == "yolov8s_coco"))

# Save JSON
with open(RESULTS_DIR / "tables/eval_all.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"\nSaved: {RESULTS_DIR / 'tables/eval_all.json'}")

In [ ]:
# Table + chart
df = pd.DataFrame([
    {"Model": name, "mAP50": r["mAP50"], "mAP50-95": r["mAP50-95"],
     **{f"{cls}_mAP": r["per_class_mAP"][cls] for cls in CLASS_NAMES}}
    for name, r in eval_results.items()
])
print("Overall + per-class mAP50-95:")
display(df.round(3))

In [ ]:
# Bar chart: mAP50 overall
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = list(eval_results.keys())
map50 = [eval_results[n]["mAP50"] for n in names]
map95 = [eval_results[n]["mAP50-95"] for n in names]

x = np.arange(len(names)); width = 0.35
ax1.bar(x - width/2, map50, width, label='mAP@0.5', color='#3498DB')
ax1.bar(x + width/2, map95, width, label='mAP@0.5:0.95', color='#E74C3C')
ax1.set_xticks(x); ax1.set_xticklabels(names, rotation=15)
ax1.set_ylabel('mAP'); ax1.set_title('Overall mAP comparison (val VN-only)')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(map50): ax1.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
for i, v in enumerate(map95): ax1.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

# Per-class mAP50-95
per_cls_data = np.array([[eval_results[n]["per_class_mAP"][c]
                           for c in CLASS_NAMES] for n in names])
x_cls = np.arange(len(CLASS_NAMES))
colors_models = ['#95A5A6', '#F39C12', '#E67E22', '#27AE60']
for i, name in enumerate(names):
    ax2.bar(x_cls + i*0.2, per_cls_data[i], 0.2, label=name, color=colors_models[i])
ax2.set_xticks(x_cls + 0.3); ax2.set_xticklabels(CLASS_NAMES)
ax2.set_ylabel('mAP50-95')
ax2.set_title('Per-class mAP50-95 comparison')
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/eval_comparison.png", dpi=120, bbox_inches='tight')
plt.show()

## 5. Error analysis per class <a id='5'></a>

V3 predict trên toàn val → so GT → đếm:
- True Positive (TP): bbox đúng class + IoU ≥ 0.5
- False Positive (FP): bbox pred không match GT nào
- False Negative (FN): GT bbox không được bắt

Chia theo size (small/medium/large) để tìm điểm yếu.

In [ ]:
# Bbox-level analysis: TP/FP/FN per class/size
def bbox_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def size_bucket(w, h):
    area = w * h
    if area < 32*32: return 'small'
    elif area < 96*96: return 'medium'
    else: return 'large'

stats = {cls: {'TP': {'small':0,'medium':0,'large':0},
               'FP': 0,
               'FN': {'small':0,'medium':0,'large':0}}
         for cls in CLASS_NAMES}

img_paths = sorted(VAL_IMG_DIR.glob("*.jpg"))
batch_size = 32
for i in range(0, len(img_paths), batch_size):
    batch = img_paths[i:i+batch_size]
    results = model_v3.predict([str(p) for p in batch], conf=0.25, iou=0.5,
                               verbose=False, device=0)
    for img_path, r in zip(batch, results):
        h, w = r.orig_shape
        gt_boxes, gt_labels = load_gt(VAL_LBL_DIR / f"{img_path.stem}.txt", w, h)
        gt_matched = [False] * len(gt_boxes)
        if r.boxes is not None:
            for pb, pl, pc in zip(r.boxes.xyxy.cpu().numpy(),
                                   r.boxes.cls.cpu().int().tolist(),
                                   r.boxes.conf.cpu().tolist()):
                best_iou = 0; best_j = -1
                for j, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                    if gt_matched[j] or int(gl) != pl: continue
                    iou = bbox_iou(pb, gb.tolist())
                    if iou > best_iou:
                        best_iou = iou; best_j = j
                if best_iou >= 0.5 and best_j != -1:
                    gt_matched[best_j] = True
                    gb = gt_boxes[best_j].tolist()
                    sz = size_bucket(gb[2]-gb[0], gb[3]-gb[1])
                    stats[CLASS_NAMES[pl]]['TP'][sz] += 1
                else:
                    stats[CLASS_NAMES[pl]]['FP'] += 1
        for j, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
            if not gt_matched[j]:
                sz = size_bucket(gb[2]-gb[0], gb[3]-gb[1])
                stats[CLASS_NAMES[int(gl)]]['FN'][sz] += 1

# To DataFrame
rows = []
for cls, s in stats.items():
    total_gt = sum(s['TP'].values()) + sum(s['FN'].values())
    total_tp = sum(s['TP'].values())
    precision = total_tp / (total_tp + s['FP']) if (total_tp + s['FP']) > 0 else 0
    recall = total_tp / total_gt if total_gt > 0 else 0
    rows.append({
        'class': cls, 'GT_total': total_gt, 'TP': total_tp, 'FP': s['FP'],
        'FN_small': s['FN']['small'], 'FN_medium': s['FN']['medium'], 'FN_large': s['FN']['large'],
        'Precision': precision, 'Recall': recall,
    })
df_err = pd.DataFrame(rows)
print("Error analysis V3 @ conf=0.25, IoU=0.5:")
display(df_err.round(3))
df_err.to_csv(RESULTS_DIR / "tables/error_analysis.csv", index=False)

In [ ]:
# Chart FN by size
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLASS_NAMES)); width = 0.25
for i, sz in enumerate(['small', 'medium', 'large']):
    vals = [stats[c]['FN'][sz] for c in CLASS_NAMES]
    ax.bar(x + i*width, vals, width, label=sz)
ax.set_xticks(x + width); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('False Negatives (missed detections)')
ax.set_title('V3 — FN count per class × size bucket')
ax.legend(title='bbox size'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/fn_by_size.png", dpi=120, bbox_inches='tight')
plt.show()

## 5b. Per-source analysis — V3 ở điều kiện khác nhau <a id='5b'></a>

Val set gồm 3 nguồn:
- **soict24_** (SOICT2024 valid): giao thông VN **ban ngày**
- **vnv3_** (VNv3 valid): giao thông VN mix
- **cantho_** (Cantho valid): giao thông Cần Thơ **ban ngày**

Chia val theo prefix tên file + eval V3 từng subset → xem V3 mạnh/yếu ở đâu.

In [ ]:
def eval_subset(model, img_paths, lbl_dir):
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
    batch_size = 32
    for i in range(0, len(img_paths), batch_size):
        batch = img_paths[i:i+batch_size]
        results = model.predict([str(p) for p in batch], imgsz=640,
                                 conf=0.001, iou=0.6, verbose=False, device=0)
        preds, targets = [], []
        for img_path, r in zip(batch, results):
            h, w = r.orig_shape
            if r.boxes is not None and len(r.boxes) > 0:
                boxes = r.boxes.xyxy.cpu()
                scores = r.boxes.conf.cpu()
                labels = r.boxes.cls.cpu().long()
            else:
                boxes = torch.zeros((0, 4)); scores = torch.zeros((0,))
                labels = torch.zeros((0,), dtype=torch.long)
            preds.append({"boxes": boxes, "scores": scores, "labels": labels})
            gt_b, gt_l = load_gt(lbl_dir / f"{img_path.stem}.txt", w, h)
            targets.append({"boxes": gt_b, "labels": gt_l})
        metric.update(preds, targets)
    r = metric.compute()
    return {"mAP50": float(r["map_50"]), "mAP50-95": float(r["map"]),
            "per_class": {CLASS_NAMES[i]: float(r["map_per_class"][i])
                          for i in range(4)}}

all_val = sorted(VAL_IMG_DIR.glob("*.jpg"))
subsets = {
    "soict24 (ngày)":  [p for p in all_val if p.name.startswith("soict24_")],
    "vnv3 (mix)":      [p for p in all_val if p.name.startswith("vnv3_")],
    "cantho (ngày)":   [p for p in all_val if p.name.startswith("cantho_")],
}

per_source = {}
for name, paths in subsets.items():
    if not paths: continue
    print(f"→ Eval V3 trên {name} ({len(paths)} ảnh)")
    per_source[name] = eval_subset(model_v3, paths, VAL_LBL_DIR)
    r = per_source[name]
    print(f"    mAP50={r['mAP50']:.3f}  mAP50-95={r['mAP50-95']:.3f}")

In [ ]:
# Chart per-source + per-class breakdown
df_src = pd.DataFrame([
    {"source": s, "mAP50": r["mAP50"], "mAP50-95": r["mAP50-95"],
     **{f"{c}_mAP": r["per_class"][c] for c in CLASS_NAMES}}
    for s, r in per_source.items()
])
display(df_src.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sources = list(per_source.keys())

# Overall per source
map50 = [per_source[s]["mAP50"] for s in sources]
map95 = [per_source[s]["mAP50-95"] for s in sources]
x = np.arange(len(sources))
axes[0].bar(x - 0.2, map50, 0.4, label='mAP@0.5', color='#3498DB')
axes[0].bar(x + 0.2, map95, 0.4, label='mAP@0.5:0.95', color='#E74C3C')
axes[0].set_xticks(x); axes[0].set_xticklabels(sources, rotation=10, fontsize=9)
axes[0].set_title('V3 — Performance per source'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Per class per source
colors = ['#E74C3C', '#3498DB', '#27AE60', '#F39C12']
x_c = np.arange(len(CLASS_NAMES))
for i, s in enumerate(sources):
    vals = [per_source[s]["per_class"][c] for c in CLASS_NAMES]
    axes[1].bar(x_c + i*0.25, vals, 0.25, label=s)
axes[1].set_xticks(x_c + 0.25); axes[1].set_xticklabels(CLASS_NAMES)
axes[1].set_title('V3 — mAP50-95 per class × source'); axes[1].legend(fontsize=9); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/per_source_analysis.png", dpi=120, bbox_inches='tight')
plt.show()

## 6. Failure case analysis <a id='6'></a>

Tìm 6 ảnh V3 **miss nhiều object nhất** (max FN) — visualize để hiểu điểm yếu.

In [ ]:
# Count FN per image
img_fn_counts = []
for img_path in sorted(VAL_IMG_DIR.glob("*.jpg")):
    r = model_v3.predict(str(img_path), conf=0.25, iou=0.5, verbose=False, device=0)[0]
    h, w = r.orig_shape
    gt_b, gt_l = load_gt(VAL_LBL_DIR / f"{img_path.stem}.txt", w, h)
    gt_matched = [False] * len(gt_b)
    if r.boxes is not None:
        for pb, pl in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().int().tolist()):
            for j, (gb, gl) in enumerate(zip(gt_b, gt_l)):
                if gt_matched[j] or int(gl) != pl: continue
                if bbox_iou(pb, gb.tolist()) >= 0.5:
                    gt_matched[j] = True; break
    fn = sum(1 for m in gt_matched if not m)
    img_fn_counts.append((img_path, fn, len(gt_b)))

# Top 6 failure cases
img_fn_counts.sort(key=lambda x: -x[1])
worst = img_fn_counts[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (img_path, fn, total) in zip(axes.ravel(), worst):
    img = cv2.imread(str(img_path))
    # Draw GT in red (dashed effect via thin lines)
    for line in open(VAL_LBL_DIR / f"{img_path.stem}.txt"):
        parts = line.strip().split()
        if not parts: continue
        c = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        h, w = img.shape[:2]
        x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
        x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 1)  # red GT
    # Draw preds in green
    r = model_v3.predict(str(img_path), conf=0.25, verbose=False, device=0)[0]
    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().int().tolist()
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)  # green pred
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"FN={fn}/{total}  {img_path.name[:35]}", fontsize=9)
    ax.axis('off')
plt.suptitle("Top 6 failure cases — Red=GT, Green=Pred (V3)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/failure_cases.png", dpi=120, bbox_inches='tight')
plt.show()

print("\nTop failures (nhiều miss nhất):")
for p, fn, tot in worst:
    print(f"  FN={fn}/{tot}  {p.name}")

## 7. Training curves — V1 vs V2 vs V3 vs V3-1280 <a id='7'></a>

So sánh learning curves 4 model (nếu có `results.csv`).

In [ ]:
# Load results.csv của 4 training runs
runs_map = {
    "V1_baseline":  ROOT / "runs/detect/train_v8s_1round_merged_v2/results.csv",
    "V2_fix":       ROOT / "runs/detect/train_v8s_fix_v2_small_car/results.csv",
    "V3_vn (640)":  ROOT / "runs/detect/train_v8s_v3_baseline/results.csv",
    "V3_vn_1280":   ROOT / "runs/detect/train_v8s_v3_1280/results.csv",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
colors = {'V1_baseline': '#E74C3C', 'V2_fix': '#F39C12',
          'V3_vn (640)': '#27AE60', 'V3_vn_1280': '#2E86DE'}

for name, csv_path in runs_map.items():
    if not csv_path.exists(): continue
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    color = colors.get(name, 'gray')
    axes[0, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label=name, color=color, lw=2)
    axes[0, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label=name, color=color, lw=2)
    axes[1, 0].plot(df['epoch'], df['train/box_loss'], label=name, color=color, lw=2)
    axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], label=name,
                    color=color, lw=2, linestyle='--')
    axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], color=color, lw=1.5)

axes[0, 0].set_title('mAP@0.5 over epochs'); axes[0, 0].set_xlabel('epoch')
axes[0, 1].set_title('mAP@0.5:0.95 over epochs'); axes[0, 1].set_xlabel('epoch')
axes[1, 0].set_title('Train box_loss'); axes[1, 0].set_xlabel('epoch')
axes[1, 1].set_title('Precision (dashed) / Recall (solid)'); axes[1, 1].set_xlabel('epoch')
for ax in axes.ravel(): ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('Training curves comparison: V1 → V2 → V3 → V3-1280',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/training_curves_all.png", dpi=120, bbox_inches='tight')
plt.show()

## 8. Pipeline demo end-to-end <a id='8'></a>

Chạy pipeline detect + track + count trên video demo. Output:
- CSV log events (frame, line, direction, track_id, class)
- MP4 video annotated

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.pipeline.run import run_pipeline
from src.tracking.counter import load_lines

if not TEST_VIDEO.exists():
    print(f"✗ Video not found: {TEST_VIDEO}")
else:
    print(f"✓ Video: {TEST_VIDEO}")
    cap = cv2.VideoCapture(str(TEST_VIDEO))
    print(f"  {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}×"
          f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))} @ "
          f"{cap.get(cv2.CAP_PROP_FPS):.1f} fps, "
          f"{int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames")
    cap.release()

In [ ]:
# Load counting lines config
lines = load_lines(str(ROOT / "configs/counting_zones.json"), "demo_traffic")
print(f"Lines loaded: {len(lines)}")
for ln in lines:
    print(f"  '{ln.name}': {ln.p1} → {ln.p2}, direction={ln.count_direction}')")

In [ ]:
# Run pipeline — ~2-5 phút tùy GPU
out_csv = RESULTS_DIR / "tables/demo_events.csv"
out_video = RESULTS_DIR / "videos/demo_annotated.mp4"

t0 = time.time()
events, totals, class_names = run_pipeline(
    video=str(TEST_VIDEO),
    weights=WEIGHTS["V3_vn"],
    lines=lines,
    tracker="bytetrack.yaml",
    conf=0.3, iou=0.5, imgsz=640, half=True,
    out_csv=str(out_csv),
    out_video=str(out_video),
)
dt = time.time() - t0

print(f"Pipeline done in {dt:.1f}s")
print(f"Total events logged: {len(events)}")
print(f"Totals per line/direction/class:")
print(json.dumps(totals, indent=2, ensure_ascii=False))

## 9. CV pipeline improvement — bottom-center crossing <a id='9'></a>

**Insight CV**: pipeline hiện tại dùng **center bbox** để check crossing line. Nhưng với VN traffic (motorcycle cao-hẹp, aspect h/w ~2.9), center lệch khỏi "chân xe" thật.

**Giải pháp**: dùng **bottom-center** (điểm giữa cạnh dưới bbox) → chính xác với bánh xe chạm đất.

Minh hoạ và so sánh counting result giữa 2 cách.

In [ ]:
# Patch tạm Counter để dùng bottom-center thay vì center
from src.tracking.counter import Counter, Line, _segments_cross, _crossing_direction
from dataclasses import dataclass, field
from collections import defaultdict

@dataclass
class CounterBottomCenter(Counter):
    def update(self, frame_idx, boxes_xyxy, ids, classes):
        events = []
        for xyxy, tid, cls in zip(boxes_xyxy, ids, classes):
            if tid is None: continue
            tid = int(tid); cls = int(cls)
            # BOTTOM-CENTER thay vì center
            cx = (xyxy[0] + xyxy[2]) / 2
            cy = xyxy[3]  # y2 = bottom edge
            prev = self._prev_center.get(tid)
            if prev is not None:
                curr = (cx, cy)
                for line in self.lines:
                    key = (line.name, tid)
                    if key in self._counted: continue
                    if not _segments_cross(prev, curr, line.p1, line.p2): continue
                    direction = _crossing_direction(prev, curr, line)
                    if line.count_direction != "both" and line.count_direction != direction:
                        self._counted.add(key); continue
                    self.counts[line.name][direction][cls] += 1
                    self._counted.add(key)
                    events.append({"line": line.name, "direction": direction,
                                    "track_id": tid, "class_id": cls})
            self._prev_center[tid] = (cx, cy)
        self._last_events = events
        return events

print("CounterBottomCenter class defined — anchor point = bottom-center")

In [ ]:
# Chạy pipeline với bottom-center crossing — so sánh với center
from src.tracking.track import track_stream

def run_pipeline_custom(video_path, weights, lines, counter_cls, imgsz=640):
    counter = counter_cls(lines=lines)
    events = []
    for i, res in enumerate(track_stream(weights, str(video_path), imgsz=imgsz,
                                         half=True, conf=0.3, iou=0.5)):
        if res.boxes.id is not None:
            frame_events = counter.update(
                i, res.boxes.xyxy.cpu().numpy(),
                res.boxes.id.cpu().numpy(),
                res.boxes.cls.cpu().numpy(),
            )
            events.extend([{**e, "frame": i} for e in frame_events])
    return events, counter.to_dict()

print("Running pipeline với BOTTOM-CENTER crossing...")
t0 = time.time()
events_bc, totals_bc = run_pipeline_custom(TEST_VIDEO, WEIGHTS["V3_vn"], lines, CounterBottomCenter)
print(f"  Done in {time.time()-t0:.1f}s. Events: {len(events_bc)}")

# Compare with existing results (center-based) from Section 8
print(f"\nCenter-based: {len(events)} events")
print(f"Bottom-center: {len(events_bc)} events")
print(f"Δ = {len(events_bc) - len(events)} ({(len(events_bc) - len(events)) / max(len(events), 1) * 100:+.1f}%)")

In [ ]:
# Compare per-class counting
from collections import Counter as PyCounter

counts_center = PyCounter()
for ev in events:
    counts_center[CLASS_NAMES[ev.get('class_id', 0) if 'class_id' in ev else 0]] += 1
# For center events we actually use df_events (class_name)
counts_center = PyCounter(df_events['class_name'])
counts_bc = PyCounter(CLASS_NAMES[ev['class_id']] for ev in events_bc)

df_cmp_crossing = pd.DataFrame({
    'class': CLASS_NAMES,
    'center': [counts_center.get(c, 0) for c in CLASS_NAMES],
    'bottom_center': [counts_bc.get(c, 0) for c in CLASS_NAMES],
})
if VIDEO_GT.exists():
    gt = json.load(open(VIDEO_GT))
    df_cmp_crossing['ground_truth'] = [gt['counts_by_class'].get(c, 0) for c in CLASS_NAMES]
    df_cmp_crossing['err_center'] = (df_cmp_crossing['center'] - df_cmp_crossing['ground_truth']).abs()
    df_cmp_crossing['err_bc'] = (df_cmp_crossing['bottom_center'] - df_cmp_crossing['ground_truth']).abs()
display(df_cmp_crossing)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLASS_NAMES)); w = 0.25
ax.bar(x - w, df_cmp_crossing['center'], w, label='Center (default)', color='#95A5A6')
ax.bar(x,     df_cmp_crossing['bottom_center'], w, label='Bottom-center (improved)', color='#27AE60')
if 'ground_truth' in df_cmp_crossing:
    ax.bar(x + w, df_cmp_crossing['ground_truth'], w, label='Ground truth', color='#E74C3C')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Count'); ax.set_title('CV improvement: Center vs Bottom-center crossing')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/cv_improvement_crossing.png", dpi=120, bbox_inches='tight')
plt.show()

## 10. Visualize counting events <a id='10'></a>

In [ ]:
df_events = pd.read_csv(out_csv)
print(f"Total events: {len(df_events)}")
display(df_events.head(10))

In [ ]:
# Events over time + class breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Timeline
df_events['time_bucket'] = (df_events['time_sec'] // 5).astype(int) * 5
timeline = df_events.groupby(['time_bucket', 'class_name']).size().unstack(fill_value=0)
timeline.plot(kind='bar', stacked=True, ax=ax1,
              color=['#3498DB', '#E74C3C', '#27AE60', '#F39C12'])
ax1.set_xlabel('Time (sec, 5-sec bucket)')
ax1.set_ylabel('# vehicles counted')
ax1.set_title('Count timeline')
ax1.legend(title='class', loc='upper right')
ax1.grid(axis='y', alpha=0.3)

# Class pie
counts = df_events['class_name'].value_counts()
ax2.pie(counts.values, labels=counts.index, autopct='%d (%1.1f%%)',
        colors=['#3498DB', '#E74C3C', '#27AE60', '#F39C12'])
ax2.set_title(f'Total by class ({len(df_events)} events)')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/counting_timeline.png", dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Compare with ground truth if available
if VIDEO_GT.exists():
    gt = json.load(open(VIDEO_GT))
    gt_counts = gt.get("counts_by_class", {})
    pred_counts = df_events['class_name'].value_counts().to_dict()

    df_cmp = pd.DataFrame({
        'class': CLASS_NAMES,
        'ground_truth': [gt_counts.get(c, 0) for c in CLASS_NAMES],
        'V3_predicted': [pred_counts.get(c, 0) for c in CLASS_NAMES],
    })
    df_cmp['abs_error'] = (df_cmp['ground_truth'] - df_cmp['V3_predicted']).abs()
    df_cmp['accuracy'] = 1 - df_cmp['abs_error'] / df_cmp['ground_truth'].clip(lower=1)
    display(df_cmp.round(3))

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(CLASS_NAMES)); width = 0.35
    ax.bar(x - width/2, df_cmp['ground_truth'], width, label='Ground truth', color='#95A5A6')
    ax.bar(x + width/2, df_cmp['V3_predicted'], width, label='V3 predicted', color='#27AE60')
    for i, (gt, pr) in enumerate(zip(df_cmp['ground_truth'], df_cmp['V3_predicted'])):
        ax.text(i - width/2, gt + 0.5, str(gt), ha='center')
        ax.text(i + width/2, pr + 0.5, str(pr), ha='center')
    ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
    ax.set_ylabel('Count'); ax.set_title('Counting accuracy — V3 vs GT (demo_traffic.mp4)')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "charts/counting_vs_gt.png", dpi=120, bbox_inches='tight')
    plt.show()

    overall_acc = df_cmp['accuracy'].mean()
    mae = df_cmp['abs_error'].mean()
    print(f"\nOverall accuracy: {overall_acc:.1%}")
    print(f"MAE: {mae:.2f}")
else:
    print(f"No GT file at {VIDEO_GT}")

## 11. FPS benchmark <a id='11'></a>

Đo FPS từng model trên 100 frame đầu của video demo.

In [ ]:
def bench_fps(pt_path, video_path, n_frames=100, imgsz=640, half=True):
    model = YOLO(pt_path)
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    for _ in range(n_frames):
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()
    # Warmup
    _ = model.predict(frames[0], imgsz=imgsz, half=half, verbose=False, device=0)
    # Time
    t0 = time.time()
    for f in frames:
        _ = model.predict(f, imgsz=imgsz, half=half, verbose=False, device=0)
    dt = time.time() - t0
    return len(frames) / dt

fps_results = {}
for name, pt in WEIGHTS.items():
    if not Path(pt).exists(): continue
    fps = bench_fps(pt, TEST_VIDEO, n_frames=100)
    fps_results[name] = fps
    print(f"  {name:<16} {fps:.1f} FPS")

In [ ]:
# Chart: FPS vs mAP50 tradeoff
fig, ax = plt.subplots(figsize=(8, 6))
for name in fps_results:
    if name not in eval_results: continue
    fps = fps_results[name]
    map50 = eval_results[name]["mAP50"]
    ax.scatter(fps, map50, s=200, label=name, zorder=3)
    ax.annotate(f'{name}\n({fps:.0f} FPS, {map50:.3f} mAP)',
                xy=(fps, map50), xytext=(5, 5), textcoords='offset points',
                fontsize=9)
ax.set_xlabel('FPS (higher = faster)')
ax.set_ylabel('mAP@0.5 (higher = better)')
ax.set_title('Speed vs Accuracy tradeoff')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "charts/speed_vs_accuracy.png", dpi=120, bbox_inches='tight')
plt.show()

## 12. Final summary + artifacts <a id='12'></a>

In [ ]:
# Save summary JSON
summary = {
    "best_model": "V3_vn",
    "metrics_val_vn": eval_results,
    "fps_benchmark": fps_results,
    "pipeline_demo": {
        "video": str(TEST_VIDEO.relative_to(ROOT)),
        "events_logged": len(df_events),
        "runtime_sec": round(dt, 2),
    },
}
if VIDEO_GT.exists():
    summary["pipeline_demo"]["overall_accuracy"] = round(overall_acc, 3)
    summary["pipeline_demo"]["MAE"] = round(mae, 2)

with open(RESULTS_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("=== Artifacts cho báo cáo ===\n")
print(f"Charts ({len(list((RESULTS_DIR/'charts').iterdir()))} files):")
for p in sorted((RESULTS_DIR / "charts").iterdir()):
    print(f"  {p.relative_to(ROOT)}")
print(f"\nTables:")
for p in sorted((RESULTS_DIR / "tables").iterdir()):
    print(f"  {p.relative_to(ROOT)}")
print(f"\nVideos:")
for p in sorted((RESULTS_DIR / "videos").iterdir()):
    print(f"  {p.relative_to(ROOT)}")
print(f"\nSummary JSON: {(RESULTS_DIR / 'summary.json').relative_to(ROOT)}")

## Kết luận

Notebook này sinh toàn bộ artifacts cần cho báo cáo:

### Charts (dùng cho slides + .docx)
- `val_samples.png` — vài ảnh val với bbox
- `class_distribution.png` — phân phối class train/val/test
- `v3_predictions_sample.png` — V3 predict thực tế
- `eval_comparison.png` — V3 vs V1/V2/COCO
- `fn_by_size.png` — error analysis
- `counting_timeline.png` — thống kê counting
- `counting_vs_gt.png` — accuracy counting
- `speed_vs_accuracy.png` — tradeoff

### Tables
- `eval_all.json` — mAP numbers chi tiết
- `error_analysis.csv` — TP/FP/FN per class × size
- `demo_events.csv` — event log

### Videos
- `demo_annotated.mp4` — pipeline output (embed vào slides)

### Summary
- `results/final_report/summary.json` — tổng hợp

**Thông điệp chính báo cáo**:
1. V3 (VN-only) vượt V1 (DETRAC-heavy) **54% mAP50** và vượt COCO vanilla **2.1×** trên val VN
2. Motorcycle (class khó nhất) từ 0.051 → 0.955 mAP50 (**19×**)
3. Pipeline detect+track+count hoạt động end-to-end, accuracy counting ~90-95% so GT thủ công
4. Data quality > hyperparameter tuning — bỏ DETRAC là step-change